# Browse DestinE Climate DT Data with Polytope

This notebook demonstrates how to browse and explore DestinE Climate Digital Twin data
using Polytope and `earthkit-data`, focusing on the **ICON** and **IFS-FESOM** models
with applications for rainfall analysis over Africa.

## Prerequisites
- Conda environment `destine` with `earthkit-data`, `polytope-client`, `covjsonkit` installed
- DestinE Platform account with upgraded access
- Authentication configured (`~/.polytopeapirc`)

See `docs/polytope_setup.md` for detailed setup instructions.

## 1. Imports and Configuration

In [ ]:
import earthkit.data
import os

# Polytope server configuration
POLYTOPE_ADDRESS = "polytope.mn5.apps.dte.destination-earth.eu"
COLLECTION = "destination-earth"

# Optional: set data output directory
# os.environ["EARTHKIT_DATA_DIR"] = "./data"

print(f"earthkit-data version: {earthkit.data.__version__}")
print(f"Polytope address: {POLYTOPE_ADDRESS}")

## 2. Quick Connection Test

Make a small test request to verify connectivity and authentication.

In [ ]:
# Minimal test: single time step, one parameter, IFS-NEMO
test_request = {
    'activity': 'projections',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20200102',
    'experiment': 'SSP3-7.0',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'IFS-NEMO',
    'param': '134',          # Surface pressure
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0100',
    'type': 'fc'
}

try:
    data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        test_request,
        address=POLYTOPE_ADDRESS,
        stream=False
    )
    print("✓ Connection successful!")
    print(data)
    # Convert to xarray for inspection
    ds = data.to_xarray()
    print(f"\nData shape: {ds}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("Check that you have valid credentials in ~/.polytopeapirc")

## 3. Browse Available Data

Explore data availability for different models and experiments.

### 3.1 ICON — Control Simulation (1950 forcing)

In [ ]:
# ICON control simulation — surface variables, one day
icon_control_request = {
    'activity': 'control',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20200102',
    'experiment': 'control-1950',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'ICON',
    'param': '134/165/166',    # SP, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0300/0600/0900/1200/1500/1800/2100',  # All 3-hourly steps
    'type': 'fc'
}

print("Requesting ICON control data...")
try:
    icon_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        icon_control_request,
        address=POLYTOPE_ADDRESS,
        stream=False
    )
    print(f"✓ Retrieved {len(icon_data)} fields")
    # Peek at metadata
    for i, field in enumerate(icon_data[:3]):
        print(f"  Field {i}: {field}")
except Exception as e:
    print(f"✗ Error: {e}")

### 3.2 ICON — Historical Simulation

In [ ]:
# ICON historical simulation — precipitation
icon_hist_request = {
    'activity': 'projections',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20000615',           # Mid-June (wet season in West Africa)
    'experiment': 'historical',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'ICON',
    'param': '228228/167',        # Total precipitation and 2m temperature
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/1200',
    'type': 'fc'
}

print("Requesting ICON historical precipitation & temperature...")
try:
    icon_hist_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        icon_hist_request,
        address=POLYTOPE_ADDRESS,
        stream=False
    )
    print(f"✓ Retrieved {len(icon_hist_data)} fields")
    # Convert to xarray
    ds = icon_hist_data.to_xarray()
    print(f"Dataset: {ds}")
except Exception as e:
    print(f"✗ Error: {e}")

### 3.3 IFS-FESOM — Projections (SSP3-7.0)

In [ ]:
# IFS-FESOM future projections — mid-century
fesom_proj_request = {
    'activity': 'projections',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20500615',           # Mid-century, wet season
    'experiment': 'SSP3-7.0',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'IFS-FESOM',
    'param': '228228/167/165/166',  # TP, 2T, 10U, 10V
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0600/1200/1800',
    'type': 'fc'
}

print("Requesting IFS-FESOM SSP3-7.0 projections...")
try:
    fesom_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        fesom_proj_request,
        address=POLYTOPE_ADDRESS,
        stream=False
    )
    print(f"✓ Retrieved {len(fesom_data)} fields")
except Exception as e:
    print(f"✗ Error: {e}")

### 3.4 IFS-FESOM — Control Simulation

In [ ]:
# IFS-FESOM control
fesom_control_request = {
    'activity': 'control',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20000615',
    'experiment': 'control-1950',
    'expver': '0001',
    'generation': '2',
    'levtype': 'sfc',
    'model': 'IFS-FESOM',
    'param': '228228',            # Total precipitation only
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0000/0600/1200/1800',
    'type': 'fc'
}

print("Requesting IFS-FESOM control precipitation...")
try:
    fesom_ctrl_data = earthkit.data.from_source(
        "polytope",
        COLLECTION,
        fesom_control_request,
        address=POLYTOPE_ADDRESS,
        stream=False
    )
    print(f"✓ Retrieved {len(fesom_ctrl_data)} fields")
except Exception as e:
    print(f"✗ Error: {e}")

## 4. Working with Retrieved Data

Convert retrieved data to xarray for analysis and plotting.

In [ ]:
# Example: quick visualization of retrieved data
# (requires matplotlib and cartopy)

def quick_plot(data, variable_name="Surface pressure", cmap="viridis"):
    """Quick plot of first field in the dataset."""
    try:
        import matplotlib.pyplot as plt
        import cartopy.crs as ccrs
        
        ds = data.to_xarray()
        
        fig = plt.figure(figsize=(12, 6))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.coastlines()
        ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
        
        # Plot first variable/time step
        first_var = list(ds.data_vars)[0]
        ds[first_var].isel(time=0 if 'time' in ds.dims else None).plot(
            ax=ax, cmap=cmap, transform=ccrs.PlateCarree()
        )
        ax.set_title(f"{variable_name}")
        plt.show()
    except ImportError:
        print("matplotlib/cartopy not available for plotting")
        print("Install with: conda install matplotlib cartopy")

# Uncomment to plot if data was successfully retrieved:
# if 'icon_data' in locals():
#     quick_plot(icon_data, "ICON Surface Pressure")

## 5. Low-Level Access with polytope-client

For advanced use cases, use the lower-level `polytope-client` directly.

In [ ]:
# Using the low-level polytope-client (requires explicit credentials
# or ~/.polytopeapirc)

from polytope.api import Client

# Client picks up credentials from ~/.polytopeapirc automatically
# Or pass explicitly: Client(address=..., user_email=..., user_key=...)

# client = Client(address=POLYTOPE_ADDRESS)

# Cancel any pending requests from previous sessions
# client.revoke("all")

# Download data
# files = client.retrieve(COLLECTION, request, output_path="./downloaded_data")
# print(f"Downloaded files: {files}")

print("Use the cell above with valid credentials to download via polytope-client.")
print("For most use cases, earthkit-data (sections above) is recommended.")

## 6. Request Builder Helper

A flexible function to build request dictionaries for different scenarios.

In [ ]:
def build_request(
    model: str = "ICON",
    activity: str = "control",
    experiment: str = "control-1950",
    date: str = "20200102",
    time: str = "0000",
    param: str = "134",
    levtype: str = "sfc",
    realization: str = "1",
    **kwargs
) -> dict:
    """
    Build a Polytope request dictionary.
    
    Parameters
    ----------
    model : str
        Climate model: 'ICON', 'IFS-FESOM', 'IFS-NEMO'
    activity : str
        'control' or 'projections'
    experiment : str
        'control-1950', 'historical', 'SSP3-7.0', etc.
    date : str
        Date in YYYYMMDD format
    time : str
        Time in HHMM format, e.g. '0000' or '0000/0300/.../2100'
    param : str
        Parameter codes, e.g. '134' or '134/165/166'
    levtype : str
        'sfc' (surface), 'pl' (pressure levels), 'hl' (height levels)
    realization : str
        Ensemble realization number
    **kwargs
        Additional keys to add/override
    
    Returns
    -------
    dict
        Polytope request dictionary
    """
    request = {
        'activity': activity,
        'class': 'd1',
        'dataset': 'climate-dt',
        'date': date,
        'experiment': experiment,
        'expver': '0001',
        'generation': '2',
        'levtype': levtype,
        'model': model,
        'param': param,
        'realization': realization,
        'resolution': 'standard',
        'stream': 'clte',
        'time': time,
        'type': 'fc'
    }
    request.update(kwargs)
    return request


# Example: Build a request for African rainfall
rainfall_request = build_request(
    model="ICON",
    activity="projections",
    experiment="SSP3-7.0",
    date="20500615",
    time="0000/0600/1200/1800",
    param="228228/167"
)

print("Example request for African rainfall analysis:")
for key, value in rainfall_request.items():
    print(f"  {key}: {value}")

## 7. Browsing Data Catalogue Programmatically

Use the Climate DT Explorer approach to discover available data.

In [ ]:
# The polytope-examples repository provides explorer notebooks
# that can be used to browse available data lazily.
#
# Key notebooks (clone and run locally):
# - 03_lazy_browse_portfolio.ipynb  — Monthly catalogue
# - 04_lazy_browse_portfolio_hourly.ipynb — Hourly catalogue
# - 05_variable_lookup.ipynb — Search for variables
#
# Repository: https://github.com/destination-earth-digital-twins/polytope-examples

print("For interactive data exploration, see:")
print("https://github.com/destination-earth-digital-twins/polytope-examples/tree/main/climate-dt/explorer")

## 8. Notes and Best Practices

- **Start small**: Test with a single time step and one parameter before requesting larger datasets.
- **Quota limits**: Max 2 concurrent downloads, 50 req/s. Be mindful of the server load.
- **Cache data**: Use `stream=False` to save retrieved data locally.
- **Time ranges**: Request all time steps for a given day in a single call to minimize overhead.
- **Authentication**: Keep `~/.polytopeapirc` valid. The token may expire.
- **For more info**: See `docs/polytope_usage.md` and `docs/data_catalogue.md`.